In [6]:
import pandas as pd

# Load the dataset
df = pd.read_csv('malicious_phish.csv')

# Display the first few rows
print(df.head())

# Check for missing values
print(df.isnull().sum())

# Examine the distribution of labels
print(df['type'].value_counts())

                                                 url        type
0                                   br-icloud.com.br    phishing
1                mp3raid.com/music/krizz_kaliko.html      benign
2                    bopsecrets.org/rexroth/cr/1.htm      benign
3  http://www.garage-pirenne.be/index.php?option=...  defacement
4  http://adventure-nicaragua.net/index.php?optio...  defacement
url     0
type    0
dtype: int64
type
benign        428103
defacement     96457
phishing       94111
malware        32520
Name: count, dtype: int64


In [12]:
def extract_domain_properties(domain):
    domain_length = len(domain)
    has_digits = any(char.isdigit() for char in domain)
    has_hyphen = '-' in domain
    return {'domain_length': domain_length, 'has_digits': has_digits, 'has_hyphen': has_hyphen}

df['domain_properties'] = df['domain'].apply(extract_domain_properties)

                                                  url      type  \
651186        xbox360.ign.com/objects/850/850402.html  phishing   
651187   games.teamxbox.com/xbox-360/1860/Dead-Space/  phishing   
651188     www.gamespot.com/xbox360/action/deadspace/  phishing   
651189  en.wikipedia.org/wiki/Dead_Space_(video_game)  phishing   
651190      www.angelfire.com/goth/devilmaycrytonite/  phishing   

                    domain   ip  \
651186     xbox360.ign.com  NaN   
651187  games.teamxbox.com  NaN   
651188    www.gamespot.com  NaN   
651189    en.wikipedia.org  NaN   
651190   www.angelfire.com  NaN   

                                        domain_properties  
651186  {'domain_length': 15, 'has_digits': True, 'has...  
651187  {'domain_length': 18, 'has_digits': False, 'ha...  
651188  {'domain_length': 16, 'has_digits': False, 'ha...  
651189  {'domain_length': 16, 'has_digits': False, 'ha...  
651190  {'domain_length': 17, 'has_digits': False, 'ha...  


In [2]:
import socket

def get_ip(domain):
    try:
        # Set a timeout to avoid long delays
        socket.setdefaulttimeout(3)
        ip = socket.gethostbyname(domain)
        return ip
    except socket.gaierror:
        return None
    except socket.timeout:
        return 'timeout'
    except Exception as e:
        return f'error: {str(e)}'



In [16]:
from concurrent.futures import ThreadPoolExecutor
import time
# Perform IP lookup using ThreadPoolExecutor
def parallel_ip_lookup(df, max_workers=100):
    start_time = time.time()
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        df['ip'] = list(executor.map(get_ip, df['domain']))
    print(f"IP Lookup completed in {time.time() - start_time} seconds")
    return df

# Run the IP lookup with multithreading
df = parallel_ip_lookup(df)



IP Lookup completed in 1370.2406191825867 seconds


In [18]:
# Save the results to a new file
df.to_csv('urls_with_ip_and_domain_features.csv', index=False)

In [2]:
def get_whois_info(domain):
    try:
        w = whois.whois(domain)
        return {
            'domain': domain,
            'creation_date': w.creation_date if isinstance(w.creation_date, datetime) else None,
            'registrar': w.registrar,
            'country': w.country,
            'city': getattr(w, 'city', None)
        }
    except Exception as e:
        return {
            'domain': domain,
            'creation_date': None,
            'registrar': None,
            'country': None,
            'city': None,
            'error': str(e)
        }




In [1]:
from urllib.parse import urlparse

def clean_domain(domain):
    try:
        domain = domain.strip().lower()
        if domain.startswith("http"):
            from urllib.parse import urlparse
            domain = urlparse(domain).netloc
        return domain.split(':')[0]  # Remove port if present
    except:
        return None

In [3]:
import pandas as pd
import whois
from datetime import datetime
from concurrent.futures import as_completed, ThreadPoolExecutor
import time
import os

# ----------------s----------
# CONFIGURATION
# --------------------------
batch_size = 5000
batch_start = 0     # Change this if continuing from a later batch
max_workers = 5
source_file = "urls_with_ip_and_domain_features.csv"
output_file = "whois_master.csv"  # Final merged file
log_file = "whois_progress.log"   # Tracks progress


# Load data
df = pd.read_csv(source_file)
df = df[df['ip'].notnull()]
df['domain'] = df['domain'].apply(clean_domain)
domains = df['domain'].dropna().unique().tolist()
print(df['domain'].head(10).tolist())




['br-icloud.com.br', 'mp3raid.com', 'bopsecrets.org', 'www.garage-pirenne.be', 'adventure-nicaragua.net', 'buzzfil.net', 'espn.go.com', 'yourbittorrent.com', 'www.pashminaonline.com', 'allmusic.com']


In [5]:
# Load previous progress
if os.path.exists(output_file):
    done_df = pd.read_csv(output_file)
    done_domains = set(done_df['domain'].tolist())
    print(f"✅ Already processed: {len(done_domains)} domains")
else:
    done_df = pd.DataFrame()
    done_domains = set()

remaining_domains = [d for d in domains if d not in done_domains]
print(f"🕒 Domains remaining: {len(remaining_domains)}")

# --------------------------
# Process in batches
# --------------------------
for batch_start in range(0, len(remaining_domains), batch_size):
    batch_end = min(batch_start + batch_size, len(remaining_domains))
    current_batch = remaining_domains[batch_start:batch_end]
    print(f"\n🚀 Processing batch {batch_start}-{batch_end} ...")

    start_time = time.time()
    results = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_domain = {executor.submit(get_whois_info, domain): domain for domain in current_batch}
        for future in as_completed(future_to_domain):
            result = future.result()
            results.append(result)

    batch_df = pd.DataFrame(results)
    
    # Append to master file
    if os.path.exists(output_file):
        batch_df.to_csv(output_file, mode='a', header=False, index=False)
    else:
        batch_df.to_csv(output_file, index=False)

    print(f"✅ Batch {batch_start}-{batch_end} done in {round((time.time() - start_time)/60, 2)} minutes")
    
    # Save progress
    with open(log_file, 'a') as f:
        f.write(f"Batch {batch_start}-{batch_end} completed\n")

✅ Already processed: 125643 domains
🕒 Domains remaining: 1

🚀 Processing batch 0-1 ...
✅ Batch 0-1 done in 0.0 minutes


In [6]:
df_main = pd.read_csv("urls_with_ip_and_domain_features.csv")
whois_df = pd.read_csv("whois_master.csv")

final_df = pd.merge(df_main, whois_df, on='domain', how='left')
final_df.to_csv("urls_full_with_ip_domain_and_whois.csv", index=False)